In [1]:
import os
import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patheffects as pe
import seaborn as sns
from rapidfuzz import fuzz, process
from sklearn.model_selection import cross_validate
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed

from scholarlm.utils import (
    load_and_process_results,
    match_datasets,
    matching_precision_recall,
    get_filenames_in_directory,
    fit_temperature,
    apply_temperature,
    fit_temperature_from_probs,
    apply_temperature_from_probs,
)
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

/projectnb/mcnet/kevin/coastal/scholarlm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Ground Truth Dataset

In [2]:
# ---------------------------------
# Load from ground truth dataset
# ---------------------------------

# Directory
with open(os.path.join("../data/nfix/directory.json"), "r") as f:
    paper_info = json.load(f)

def text_or_table_extraction(location):
    if 'figure' in location:
        return False
    if 'supplement' in location:
        return False
    if 'archive' in location:
        return False
    if 'author' in location:
        return False
    else:
        return True

registered_paper_info = {R: Rinfo for R,Rinfo in paper_info.items() if text_or_table_extraction(Rinfo['extraction_location'])}

paper_subset = [
    "R163", "R164", "R172", "R248", "R124",
    "R51", "R59", "R114", "R43", "R103",
]

#registered_paper_info = {R: Rinfo for R,Rinfo in paper_info.items() if R in paper_subset}
registered_ids = list(registered_paper_info.keys())

ground_truth_df = pd.read_csv("../data/nfix/meta/aquatic_N2fix_rates.csv")
ground_truth_df = ground_truth_df.loc[ground_truth_df.reference_id.isin(registered_ids)]

id_cols = [
    'nfix_rate_id', 'reference_id', 'site_name', 'latitude', 'longitude',
    'habitat', 'year', 'month', 'day', 'hour_minute', 'season',
    'substrate', 'substrate_details'
]

# Build a list of records, one per attribute
records = []

# 1) nfix_rate_original -> attribute="rate_original"
#    error from nfix_error_original, error_type from nfix_error_type,
#    units from nfix_unit_original
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_rate',
    value=ground_truth_df['nfix_rate_original'],
    error=ground_truth_df['nfix_error_original'],
    error_type=ground_truth_df['nfix_error_type'],
    units=ground_truth_df['nfix_unit_original'],
))

# 2) sample_depth -> attribute="sample_depth"
records.append(ground_truth_df[id_cols].assign(
    attribute='sample_depth',
    value=ground_truth_df['sample_depth'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

# 3) nfix_method -> attribute="method"
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_method',
    value=ground_truth_df['nfix_method'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

'''
# 4) nfix_incubation_time -> attribute="incubation_time"
records.append(nfix_df[id_cols].assign(
    attribute='incubation_time',
    value=nfix_df['nfix_incubation_time'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))
'''

# 5) nfix_incubation_temp -> attribute="incubation_temp"
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_incubation_temp_temperature',
    value=ground_truth_df['nfix_incubation_temp'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

ground_truth_df = pd.concat(records, ignore_index=True)

# Reorder columns
ground_truth_df = ground_truth_df[id_cols + ['attribute', 'value', 'error', 'error_type', 'units']]
ground_truth_df = ground_truth_df.dropna(subset=['value'])

# Optionally sort so each original row's attributes are grouped together
ground_truth_df = ground_truth_df.sort_values(id_cols, kind='mergesort').reset_index(drop=True)

ground_truth_df = ground_truth_df.loc[ground_truth_df.attribute == 'nfix_rate']
ground_truth_df.reset_index(drop=True, inplace=True)

In [15]:
ground_truth_df.reference_id.value_counts()[:10]

reference_id
R163    121
R164     84
R172     42
R248     32
R124     31
R51      31
R59      26
R114     26
R43      25
R103     23
Name: count, dtype: int64

In [3]:
ground_truth_df

,nfix_rate_id,reference_id,site_name,latitude,longitude,habitat,year,month,day,hour_minute,season,substrate,substrate_details,attribute,value,error,error_type,units
0,N1,R231,south west coast of Australia,-28.000000,114.000000,continental shelves,2003.0,10.0,NaN,NaN,NaN,wc,water,nfix_rate,0.017,0.006,stdev.,nmol-n l-1 h-1
1,N1027,R25,"East Weeks Bay, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,1.6,0.200,std. err,umol-n m-2 h-1
2,N1028,R25,"Weeks Bay Mouth, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,2.4,0.700,std. err,umol-n m-2 h-1
3,N1029,R25,"West Weeks Bay, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,2.6,0.400,std. err,umol-n m-2 h-1
4,N1030,R109,"Pearl River Estuary, South China Sea",22.747341,113.652219,estuaries,2018.0,7.0,31.0,NaN,NaN,benthos,sediment - inner estuary,nfix_rate,0.19,0.110,stdev.,ug-N g-1 d-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,N983,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1990.0,7.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,22.7,NaN,NaN,nmol-c2h4 cm-2 h-1
987,N984,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1990.0,11.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,7.3,NaN,NaN,nmol-c2h4 cm-2 h-1
988,N985,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1991.0,3.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,20.3,NaN,NaN,nmol-c2h4 cm-2 h-1
989,N986,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1991.0,7.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,5.2,NaN,NaN,nmol-c2h4 cm-2 h-1


### Full, Extracted Dataset

In [8]:
# ---------------------------------
# Load experiment results
# ---------------------------------

#experiment_data_path = "../data/experiments/2026_04_01/nfix_final.json"
experiment_data_path = "../data/experiments/nfix/extraction/gpt-oss-120b/2026_04_12/final.json"

unit_conversion_table = {
    'nfix_rate': {},
    'nfix_incubation_time': {},
    'nfix_incubation_temperature': {"°C": 1,},
}

attribute_types = {
    'nfix_rate_mass': float,
    'nfix_rate_areal': float,
    'nfix_rate_volumetric': float,
    'nfix_incubation_time': float,
    'nfix_incubation_temperature': float,
}

# NOTE: some of these things you should get rid of in your extraction process!
drop_keys = ["feature_terms", "attribute_terms", "abbreviations", "table_logprob", "page_logprob", "judgement_raw_text"]
drop_attrs = ['nfix_incubation_time', 'nfix_incubation_temperature', 'sample_depth']

extracted_df = load_and_process_results(
    json_path=experiment_data_path,
    unit_conversion_table=unit_conversion_table,
    attribute_types=attribute_types,
    drop_keys=drop_keys,
    drop_attrs=drop_attrs,
    attribute_col="attribute",
    value_col="value",
    unit_col="units",
    out_col="processed_value"
)

# NOTE you need to change this to 'attribute'
#extracted_df.rename(columns={"feature": "attribute"}, inplace=True)
extracted_df.sort_values(by=["title", "attribute"], inplace=True)
extracted_df.reset_index(drop=True, inplace=True)
extracted_df.attribute = "nfix_rate"

#xtracted_df = extracted_df.loc[extracted_df.attribute == 'nfix_rate']
#extracted_df = extracted_df.reset_index(drop=True)

#extracted_df = extracted_df.loc[extracted_df.paper_code.isin(registered_ids)]
#extracted_df.reset_index(drop=True, inplace=True)

In [9]:
extracted_df

,reference,doi,doi_data,url,publication_year,authors,title,publication,volume,pages,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,processed_value
0,Hicks and Silvester 1990 N Zealand J Mar Fresh...,10.1080/00288330.1990.9516439,NaN,http://www.tandfonline.com/doi/abs/10.1080/002...,1990,"Hicks, Brendan J.; Silvester, Warwick B.",Acetylene reduction associated with Zostera no...,New Zealand Journal of Marine and Freshwater R...,24,481-486,...,15.20,umol-C2H4 m-2 h-1,[1],[None],[None],[None],[text],[Acetylene reduction associated with Zostera n...,3,15.20
1,Hicks and Silvester 1990 N Zealand J Mar Fresh...,10.1080/00288330.1990.9516439,NaN,http://www.tandfonline.com/doi/abs/10.1080/002...,1990,"Hicks, Brendan J.; Silvester, Warwick B.",Acetylene reduction associated with Zostera no...,New Zealand Journal of Marine and Freshwater R...,24,481-486,...,24.70,μmol C2H4 m-2 h-1,[1],[None],[None],[None],[text],[Acetylene reduction associated with Zostera n...,4,24.70
2,Hicks and Silvester 1990 N Zealand J Mar Fresh...,10.1080/00288330.1990.9516439,NaN,http://www.tandfonline.com/doi/abs/10.1080/002...,1990,"Hicks, Brendan J.; Silvester, Warwick B.",Acetylene reduction associated with Zostera no...,New Zealand Journal of Marine and Freshwater R...,24,481-486,...,4.50,μmol C2H4 m-2 h-1,[1],[None],[None],[None],[text],[Acetylene reduction associated with Zostera n...,5,4.50
3,Montoya et al 2019 J Mar Res 77,10.1357/002224019828474278,NaN,https://www.ingentaconnect.com/content/10.1357...,2019,"Montoya, Joseph P.; Landrum, Jason P.; Weber, ...",Amazon River influence on nitrogen fixation in...,Journal of Marine Research,77,191-213,...,75.90,umol-N m-2 d-1,[7],[1],"[('OPC', 3)]",[n_fixation_rate_umol_m2_d1_mean],[table],"[<table number=""1"">\n<caption>Summary of integ...",760,75.90
4,Shiau et al 2017 Estuar Coast Shelf Sci 189,10.1016/j.ecss.2017.03.005,NaN,https://linkinghub.elsevier.com/retrieve/pii/S...,2017,"Shiau, Yo-Jin; Lin, Ming-Fen; Tan, Chen-Chung;...",Assessing N2 fixation in estuarine mangrove soils,"Estuarine, Coastal and Shelf Science",189,84-89,...,0.39,umol-C2H4 m-2 h-1,[4],[2],"[('Mangrove forest (Taipei, TW)', 'Dwarf Kande...",[soil_nitrogenase_activity_ambient_umol_c2h4_m...,[table],"[<table number=""2"">\n<caption>Comparison of th...",1411,0.39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2072,Kern and Darwich 2003 Amazoniana 17,NaN,NaN,https://repositorio.inpa.gov.br/handle/1/19023,2003,"Kern, J; Darwich, A",The role of periphytic N2 fixation for stands ...,Amazoniana,17,361-375,...,1.00,nmol-N2 g-1 h-1,[11],[1],"[('Paspalum_repens', 'near_Solimoes_A-B', 'dar...",[value],[table],"[<table number=""1"">\n<caption>N₂ fixation rate...",558,1.00
2073,Kern and Darwich 2003 Amazoniana 17,NaN,NaN,https://repositorio.inpa.gov.br/handle/1/19023,2003,"Kern, J; Darwich, A",The role of periphytic N2 fixation for stands ...,Amazoniana,17,361-375,...,0.60,nmol-N2 g-1 h-1,[11],[1],"[('Echinochloa_polystachya', 'near_Solimoes_A-...",[value],[table],"[<table number=""1"">\n<caption>N₂ fixation rate...",559,0.60
2074,Kern and Darwich 2003 Amazoniana 17,NaN,NaN,https://repositorio.inpa.gov.br/handle/1/19023,2003,"Kern, J; Darwich, A",The role of periphytic N2 fixation for stands ...,Amazoniana,17,361-375,...,0.10,nmol-N2 g-1 h-1,[11],[1],"[('Echinochloa_polystachya', 'near_Solimoes_A-...",[value],[table],"[<table number=""1"">\n<caption>N₂ fixation rate...",560,0.10
2075,Kern and Darwich 2003 Amazoniana 17,NaN,NaN,https://repositorio.inpa.gov.br/handle/1/19023,2003,"Kern, J; Darwich, A",The role of periphytic N2 fixation for stands ...,Amazoniana,17,361-375,...,1.20,nmol-N2 g-1 h-1,[11],[1],"[('Eichhornia_crassipes', 'near_Solimoes_A-B',...",[value],[table],"[<table number=""1"">\n<caption>N₂ fixation rate...",561,1.20


### Match Extractions to Ground Truth

In [10]:
# Set of attributes which must be strictly equivalent to create a match
strict_matching = {
    "reference_id": "paper_code",
    "attribute": "attribute",
    "value": "value"
}

# Set of attributes which should be 
# compared by a fuzzy matching (roughly similar) to create a match.
fuzzy_matching = {
    "site_name": "name",
    #"substrate": "substrate_type",
    #"habitat": "habitat_type",
}

# This can take a while to run if you have a lot of data, 
# since it compares every extracted row to every ground truth row.
matching, matching_recall, matching_precision = matching_precision_recall(
    ground_truth_df,
    extracted_df,
    strict_matching=strict_matching,
    fuzzy_matching=fuzzy_matching,
    fuzzy_threshold = 0.0
)

print(f"Recall: {matching_recall:.4f}")
print(f"Precision: {matching_precision:.4f}")

Recall: 0.5671
Precision: 0.2706


### Debugging

In [39]:
gt_matched = np.array([False] * ground_truth_df.shape[0])
ex_matched = np.array([False] * extracted_df.shape[0])
for gt_idx, ex_idx in matching:
    gt_matched[gt_idx] = True
    ex_matched[ex_idx] = True

unmatched_gt = np.where(~gt_matched)[0]
unmatched_ex = np.where(~ex_matched)[0]

matched_gt_df = ground_truth_df[gt_matched == True]
unmatched_gt_df = ground_truth_df[gt_matched == False]
unmatched_gt_refs = unmatched_gt_df.reference_id.value_counts().index

matched_ex_df = extracted_df[ex_matched == True]
unmatched_ex_df = extracted_df[ex_matched == False]
unmatched_ex_refs = unmatched_ex_df.paper_code.value_counts().index

In [53]:
unmatched_gt_df.reference_id.value_counts()

reference_id
R164    83
R172    38
R248    30
R51     29
R124    25
R163    24
R114    22
R59     21
R43     12
R103    10
Name: count, dtype: int64

In [ ]:
paper_subset = [
    "R163", "R164", "R172", "R248", "R124",
    "R51", "R59", "R114", "R43", "R103",
]

In [67]:
ref = "R248"
gt_ref_df = ground_truth_df.loc[ground_truth_df.reference_id == ref]
unmatched_gt_ref_df = unmatched_gt_df.loc[unmatched_gt_df.reference_id == ref]
ex_ref_df = extracted_df.loc[extracted_df.paper_code == ref]
unmatched_ex_ref_df = unmatched_ex_df.loc[unmatched_ex_df.paper_code == ref]

In [68]:
gt_ref_df

,nfix_rate_id,reference_id,site_name,latitude,longitude,habitat,year,month,day,hour_minute,season,substrate,substrate_details,attribute,value,error,error_type,units
147,N3499,R248,Fern Lake Bog,40.682785,-82.067557,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,59.0,NaN,NaN,nmol-c2h4 g-1 d-1
148,N3500,R248,Fern Lake Bog,40.682785,-82.067557,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,57.4,NaN,NaN,nmol-c2h4 g-1 d-1
149,N3501,R248,Fern Lake Bog,40.682785,-82.067557,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,318.5,NaN,NaN,nmol-c2h4 g-1 d-1
150,N3502,R248,Fern Lake Bog,40.682785,-82.067557,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,80.8,NaN,NaN,nmol-c2h4 g-1 d-1
151,N3503,R248,J. Arthur Herrick Fen,41.213913,-81.370971,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,32.0,NaN,NaN,nmol-c2h4 g-1 d-1
152,N3504,R248,J. Arthur Herrick Fen,41.213913,-81.370971,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,117.7,NaN,NaN,nmol-c2h4 g-1 d-1
153,N3505,R248,J. Arthur Herrick Fen,41.213913,-81.370971,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,100.4,NaN,NaN,nmol-c2h4 g-1 d-1
154,N3506,R248,Kent Bog,41.126565,-81.354814,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,26.7,NaN,NaN,nmol-c2h4 g-1 d-1
155,N3507,R248,Kent Bog,41.126565,-81.354814,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,11.4,NaN,NaN,nmol-c2h4 g-1 d-1
156,N3508,R248,Triangle Lake Bog,41.120772,-81.262271,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,91.8,NaN,NaN,nmol-c2h4 g-1 d-1


In [69]:
unmatched_gt_ref_df

,nfix_rate_id,reference_id,site_name,latitude,longitude,habitat,year,month,day,hour_minute,season,substrate,substrate_details,attribute,value,error,error_type,units
147,N3499,R248,Fern Lake Bog,40.682785,-82.067557,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,59.0,NaN,NaN,nmol-c2h4 g-1 d-1
148,N3500,R248,Fern Lake Bog,40.682785,-82.067557,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,57.4,NaN,NaN,nmol-c2h4 g-1 d-1
149,N3501,R248,Fern Lake Bog,40.682785,-82.067557,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,318.5,NaN,NaN,nmol-c2h4 g-1 d-1
150,N3502,R248,Fern Lake Bog,40.682785,-82.067557,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,80.8,NaN,NaN,nmol-c2h4 g-1 d-1
151,N3503,R248,J. Arthur Herrick Fen,41.213913,-81.370971,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,32.0,NaN,NaN,nmol-c2h4 g-1 d-1
153,N3505,R248,J. Arthur Herrick Fen,41.213913,-81.370971,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,100.4,NaN,NaN,nmol-c2h4 g-1 d-1
154,N3506,R248,Kent Bog,41.126565,-81.354814,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,26.7,NaN,NaN,nmol-c2h4 g-1 d-1
156,N3508,R248,Triangle Lake Bog,41.120772,-81.262271,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,91.8,NaN,NaN,nmol-c2h4 g-1 d-1
157,N3509,R248,Triangle Lake Bog,41.120772,-81.262271,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,112.1,NaN,NaN,nmol-c2h4 g-1 d-1
158,N3510,R248,Triangle Lake Bog,41.120772,-81.262271,freshwater wetlands,NaN,NaN,NaN,NaN,NaN,benthos,roots + rhizosphere soil,nfix_rate,122.3,NaN,NaN,nmol-c2h4 g-1 d-1


In [70]:
sm_ex_ref_df = ex_ref_df.iloc[:,14:]
sm_ex_ref_df

,name,ecosystem_type,latitude,longitude,date,nfix_method,substrate_type,sample_depth,entity_id,attribute,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,processed_value
0,Triangle Lake Bog,bog,NaN,NaN,None,acetylene reduction,rhizosphere,None,doc_6_entity_2,nfix_rate,407200.0,nmoles m⁻²,[0],[None],[None],[None],[text],[Associative Rhizosphere Nitrogen Fixation (Ac...,670,407200.0
1,Triangle Lake Bog,bog,NaN,NaN,None,acetylene reduction,rhizosphere,None,doc_6_entity_2,nfix_rate,260.4,x10²,[2],[2],"[('Triangle Lake Bog', 'Mean')]",[nitrogenase_activity_per_m2_x102],[table],"[<table number=""2"">\n<caption>Nitrogenase acti...",671,260.4
2,Fern Lake Bog,bog,NaN,NaN,None,acetylene reduction,rhizosphere,None,doc_6_entity_0,nfix_rate,116.5,x10² m⁻²,[2],[2],"[('Fern Lake Bog', 'Mean')]",[nitrogenase_activity_per_m2_x102],[table],"[<table number=""2"">\n<caption>Nitrogenase acti...",1127,116.5
3,Kent Bog,bog,NaN,NaN,None,acetylene reduction,rhizosphere,None,doc_6_entity_1,nfix_rate,45.9,x10^2,[2],[2],"[('Kent Bog', 'Mean')]",[nitrogenase_activity_per_m2_x102],[table],"[<table number=""2"">\n<caption>Nitrogenase acti...",1129,45.9
4,J. Arthur Herrick Fen,fen,NaN,NaN,None,acetylene reduction,rhizosphere,None,doc_6_entity_3,nfix_rate,790.2,umol-N m-2 h-1,[2],[2],"[('J. Arthur Herrick Fen', 'Mean')]",[nitrogenase_activity_per_m2_x102],[table],"[<table number=""2"">\n<caption>Nitrogenase acti...",1131,790.2
5,Fern Lake Bog,bog,NaN,NaN,None,acetylene reduction,rhizosphere,None,doc_6_entity_0,nfix_rate,330.0,nmoles N₂ g⁻¹ h⁻¹,[4],[None],[None],[None],[text],"[Klucas, R. V. 1991. Associative nitrogen fixa...",663,330.0
6,Fern Lake Bog,bog,NaN,NaN,None,acetylene reduction,rhizosphere,None,doc_6_entity_0,nfix_rate,128.9,nmol-N g-1 h-1,[2],[2],"[('Fern Lake Bog', 'Mean')]",[nitrogenase_activity_per_g_root],[table],"[<table number=""2"">\n<caption>Nitrogenase acti...",664,128.9
7,Triangle Lake Bog,bog,NaN,NaN,None,acetylene reduction,rhizosphere,None,doc_6_entity_2,nfix_rate,233.2,nmoles g Dry Mass⁻¹ 24 h⁻¹,[0],[None],[None],[None],[text],[Associative Rhizosphere Nitrogen Fixation (Ac...,668,233.2
8,Triangle Lake Bog,bog,NaN,NaN,None,acetylene reduction,rhizosphere,None,doc_6_entity_2,nfix_rate,92.5,nmol-N g-1 h-1,[2],[2],"[('Triangle Lake Bog', 'Mean')]",[nitrogenase_activity_per_g_root],[table],"[<table number=""2"">\n<caption>Nitrogenase acti...",669,92.5
9,Kent Bog,bog,NaN,NaN,None,acetylene reduction,rhizosphere,None,doc_6_entity_1,nfix_rate,11.4,nmoles g⁻¹,[2],[2],"[('Kent Bog', 'Chamaedaphne calyculata')]",[nitrogenase_activity_per_g_root],[table],"[<table number=""2"">\n<caption>Nitrogenase acti...",1128,11.4


In [58]:
len(ex_ref_df)

28

In [43]:
table1_unmatched_df = unmatched_ex_ref_df[unmatched_ex_ref_df['table_number'].apply(lambda x: 1 in x)]

In [50]:
table1_unmatched_df.loc[table1_unmatched_df.value == 3.7].iloc[:, 14:]

,name,location,latitude,longitude,habitat_type,substrate_type,sample_depth,year,month,day,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,processed_value
1098,Mid-Atlantic Shelf,"37.519°N, −75.050°W",37.519,-75.050,shelf,water,29,2006.0,7.0,NaN,...,3.7,unitless,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",704,3.7
1105,Mid-Atlantic Shelf,"38.042°N, −74.299°W",38.042,-74.299,shelf,water,None,2006.0,7.0,NaN,...,3.7,umol N m−2 d−1,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",727,3.7
1135,Mid-Atlantic Shelf,"38.042°N, −74.299°W",38.042,-74.299,shelf,water,None,2006.0,7.0,NaN,...,3.7,3.7,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",886,3.7
1149,Mid-Atlantic Shelf,"37.695°N, −75.298°W",37.695,-75.298,shelf,water,None,2006.0,10.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",935,3.7
1156,Mid-Atlantic Shelf,"37.631°N, −75.152°W",37.631,-75.152,shelf,water,None,2006.0,10.0,NaN,...,3.7,3.7,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",957,3.7
1163,Mid-Atlantic Shelf,"37.519°N, −75.050°W",37.519,-75.050,shelf,water,None,2006.0,10.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",981,3.7
1199,Mid-Atlantic Shelf,"36.554°N, −75.706°W",36.554,-75.706,shelf,water,None,2009.0,8.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1117,3.7
1206,Mid-Atlantic Shelf,"37.428°N, −75.476°W",37.428,-75.476,shelf,water,None,2009.0,8.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1142,3.7
1230,Mid-Atlantic Shelf,"36.417°N, −75.700°W",36.417,-75.700,shelf,water,None,2009.0,8.0,NaN,...,3.7,None,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1261,3.7
1237,Mid-Atlantic Shelf,"36.417°N, −75.700°W",36.417,-75.700,shelf,water,None,2009.0,8.0,NaN,...,3.7,umol N m−2 d−1,[2],[1],"[('MAS 1', 'Mid-Atlantic shelf')]",[n2_fixation],[table],"[<table number=""1"">\n<caption>Physical and che...",1285,3.7


In [46]:
table1_unmatched_df.units.value_counts()

units
nmol N L−1 d−1    28
3.7               17
umol N m−2 d−1    13
nmol N            10
umol-N m-2 d-1     5
n2_fixation        1
unitless           1
3.8                1
nmol/L/d           1
nmol L-1 d-1       1
nmol L−1 d−1       1
Name: count, dtype: int64